# CN / MCI / DEM — Gaussian Naive Bayes

활동·수면 기록만으로 사람 단위 CN/MCI/DEM을 분류하는 나이브 베이즈 실험 노트북입니다. `smoke`는 동작 확인용이며, 보고용 결과는 `full` 실행으로 생성합니다.

- 입력: activity, sleep
- 제외: MMSE, 진단명, ID, 절대 날짜·수집 프로토콜 신호
- 평가: 사람 단위 3-fold nested CV
- Validation은 예측을 먼저 동결한 뒤 역사적 참고값으로만 평가합니다.


In [ ]:
# 실행 설정
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPOSITORY_URL = "https://github.com/Pig30nidaE/Google-Ajou-AICapstone.git"

# 필요할 때만 절대 경로를 지정합니다. None이면 자동 탐색합니다.
PROJECT_ROOT_OVERRIDE = None
DATA_ROOT_OVERRIDE = None
RESULTS_ROOT_OVERRIDE = None

RUN_MODE = "smoke"  # 빠른 점검: smoke, 정식 학습: full
SEED = 20260721
N_JOBS = 1
SKIP_VALIDATION_LABELS = False


In [ ]:
# 저장소와 데이터 경로를 준비합니다. Colab에서는 저장소를 /content에 clone합니다.
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    clone_root = Path("/content/Google-Ajou-AICapstone")
    if PROJECT_ROOT_OVERRIDE is None:
        if not clone_root.exists():
            subprocess.run(["git", "clone", REPOSITORY_URL, str(clone_root)], check=True)
        elif (clone_root / ".git").is_dir():
            subprocess.run(["git", "-C", str(clone_root), "pull", "--ff-only"], check=True)
        PROJECT_ROOT = clone_root.resolve()
    else:
        PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
else:
    def find_project_root(start: Path) -> Path | None:
        for candidate in (start, *start.parents):
            if (candidate / "SangHyo/ThreeClass_NaiveBayes/run_base.py").is_file():
                return candidate
        return None

    PROJECT_ROOT = (
        Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if PROJECT_ROOT_OVERRIDE
        else find_project_root(Path.cwd())
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError("PROJECT_ROOT_OVERRIDE를 설정하세요.")

data_candidates = [
    Path(DATA_ROOT_OVERRIDE).expanduser() if DATA_ROOT_OVERRIDE else None,
    PROJECT_ROOT / "Data",
    Path("/content/drive/Shareddrives/GoogleAI_contest/Data"),
    Path("/content/drive/MyDrive/GoogleAI_contest/Data"),
]
DATA_ROOT = next((path.resolve() for path in data_candidates if path and path.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError("DATA_ROOT_OVERRIDE를 설정하세요.")

EXPERIMENT_ROOT = PROJECT_ROOT / "SangHyo/ThreeClass_NaiveBayes"
TRAINING_ROOT = DATA_ROOT / "1.Training"
VALIDATION_ROOT = DATA_ROOT / "2.Validation"
if RESULTS_ROOT_OVERRIDE:
    RESULTS_ROOT = Path(RESULTS_ROOT_OVERRIDE).expanduser().resolve()
elif IN_COLAB:
    RESULTS_ROOT = Path("/content/drive/MyDrive/SangHyo_NaiveBayes_Results")
else:
    RESULTS_ROOT = EXPERIMENT_ROOT / "training_outputs"

required = [EXPERIMENT_ROOT, TRAINING_ROOT, VALIDATION_ROOT]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("필수 경로가 없습니다:\n" + "\n".join(missing))

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_ROOT    :", DATA_ROOT)
print("RESULTS_ROOT :", RESULTS_ROOT)


In [ ]:
# GaussianNB 학습에 필요한 패키지를 설치합니다.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(EXPERIMENT_ROOT / 'requirements_colab.txt'),
], check=True)


In [ ]:
# 학습을 실행합니다. 이전 run_base.py도 지원하도록 새 결과 폴더를 함께 추적합니다.
if str(EXPERIMENT_ROOT) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_ROOT))

import importlib
import run_base

run_base = importlib.reload(run_base)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
previous_runs = {path.resolve() for path in RESULTS_ROOT.iterdir() if path.is_dir()}

run_result = run_base.main({
    'PROJECT_ROOT': PROJECT_ROOT,
    'DATA_ROOT': DATA_ROOT,
    'NAIVE_BAYES_RESULTS_ROOT': RESULTS_ROOT,
    'NAIVE_BAYES_RUN_MODE': RUN_MODE,
    'SEED': SEED,
    'NAIVE_BAYES_N_JOBS': N_JOBS,
    'NAIVE_BAYES_SKIP_VALIDATION_LABELS': SKIP_VALIDATION_LABELS,
})

if run_result is not None:
    OUTPUT_DIR = Path(run_result).resolve()
else:
    completed_runs = [
        path.resolve()
        for path in RESULTS_ROOT.iterdir()
        if path.is_dir() and (path / 'FINAL_REPORT.json').is_file()
    ]
    new_runs = [path for path in completed_runs if path not in previous_runs]
    candidates = new_runs or completed_runs
    if not candidates:
        raise RuntimeError('학습은 끝났지만 FINAL_REPORT.json 결과 폴더를 찾지 못했습니다.')
    OUTPUT_DIR = max(candidates, key=lambda path: path.stat().st_mtime)

if not (OUTPUT_DIR / 'FINAL_REPORT.json').is_file():
    raise FileNotFoundError(f'FINAL_REPORT.json이 없습니다: {OUTPUT_DIR}')
print('확정된 결과 폴더:', OUTPUT_DIR)


In [ ]:
# 핵심 결과를 확인합니다.
import json
import pandas as pd

report = json.loads((OUTPUT_DIR / 'FINAL_REPORT.json').read_text(encoding='utf-8'))
summary = report['primary_nested_repeat_summary']
rows = []
for metric in ('accuracy', 'balanced_accuracy', 'macro_f1', 'roc_auc_ovr_macro', 'cn_vs_rest_auc'):
    value = summary[metric]
    rows.append({'metric': metric, 'mean': value['mean'], 'std': value['std']})
display(pd.DataFrame(rows))

validation = report.get('validation_historical')
if validation and validation.get('evaluated'):
    print('Historical Validation:', validation['metrics'])
else:
    print('Validation label은 열지 않았습니다.')
print('결과 폴더:', OUTPUT_DIR)
